# Maps of household WASH access in Nigeria

Place this notebook beside the **wash_state_outputs** folder containing State_WASH_Prevalence.csv and State_WASH_Change_2013_2024.csv. You can also place both CSVs directly beside the notebook. Open it in JupyterLab and choose **Run → Run All Cells**.

If packages are missing, run **%pip install pandas numpy matplotlib pillow python-docx** in a code cell and restart the kernel. For automatic Word 97–2003 export, install [LibreOffice Writer](https://www.libreoffice.org/download/download-libreoffice/) once. The notebook also saves a DOCX that you can open in desktop Word and save as **Word 97–2003 Document (*.doc)**.

The notebook downloads a pinned Nigeria ADM1 GeoJSON. If the automatic download is blocked, [download the boundary file](https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1_simplified.geojson) manually and save it beside the notebook as **geoBoundaries-NGA-ADM1_simplified.geojson**.

These maps describe spatial patterns. Formal hotspot testing is a separate analysis.


## 1. Check the estimates and match the boundaries

The code checks for one estimate per state, round and indicator and verifies each change against the two endpoint estimates. The boundary label **Abuja Federal Capital Territory** is matched to **FCT**. The run stops if any of the 37 areas is missing, duplicated or unmatched.


In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib
import json
import os
import shutil
import subprocess
from tempfile import TemporaryDirectory

import matplotlib.pyplot as plt
from matplotlib.colors import Normalize, TwoSlopeNorm
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path as PlotPath
import numpy as np
import pandas as pd
from PIL import Image

ROOT = Path.cwd()
OUTPUT = ROOT / "wash_map_outputs"
BOUNDARY = ROOT / "geoBoundaries-NGA-ADM1_simplified.geojson"
URL = ("https://media.githubusercontent.com/media/wmgeolab/geoBoundaries/9469f09/"
       "releaseData/gbOpen/NGA/ADM1/geoBoundaries-NGA-ADM1_simplified.geojson")
CREDIT = "GRID3 Nigeria ADM1 via geoBoundaries gbOpen, 2022 boundaries, CC BY 4.0"
YEARS = (2013, 2018, 2024)
NAMES = {"water_basic": "Basic drinking water", "sanitation_basic": "Basic sanitation",
         "hygiene_basic": "Basic hygiene"}
DELTA = "change_2024_minus_2013_pp"

def read_result(filename):
    path = ROOT / "wash_state_outputs" / filename
    if not path.is_file():
        path = ROOT / filename
    if not path.is_file():
        raise FileNotFoundError(f"Put {filename} in wash_state_outputs or beside this notebook.")
    return pd.read_csv(path)

prevalence = read_result("State_WASH_Prevalence.csv")
change = read_result("State_WASH_Change_2013_2024.csv")
required = [(prevalence, {"state", "indicator", "survey_round", "weighted_pct", "review_flags"}),
            (change, {"state", "indicator", DELTA, "review_flag"})]
for frame, columns in required:
    if missing := columns - set(frame.columns):
        raise ValueError(f"Results table is missing columns: {sorted(missing)}")
    if frame.state.isna().any() or frame.indicator.isna().any():
        raise ValueError("A result has a blank state or indicator.")

prevalence["review"] = prevalence.review_flags.fillna("").astype(str).str.strip().ne("")
change["review"] = change.review_flag.astype(str).str.strip().str.lower().map({"true": True, "false": False})
if change.review.isna().any():
    raise ValueError("Unrecognised review flag in the change table.")
states = set(prevalence.state)
if len(states) != 37 or "FCT" not in states:
    raise ValueError("Expected 36 states and the FCT.")
p = prevalence.set_index(["state", "indicator", "survey_round"], verify_integrity=True)
c = change.set_index(["state", "indicator"], verify_integrity=True)
if set(p.index) != set(pd.MultiIndex.from_product([states, NAMES, YEARS])) or (
    set(c.index) != set(pd.MultiIndex.from_product([states, NAMES]))
):
    raise ValueError("The state, indicator or year keys are incomplete or inconsistent.")
if not p.weighted_pct.dropna().between(0, 100).all():
    raise ValueError("Prevalence must be on the 0–100 scale.")
first = p.xs(2013, level="survey_round").reindex(c.index)
last = p.xs(2024, level="survey_round").reindex(c.index)
if not np.allclose(c[DELTA], last.weighted_pct - first.weighted_pct, atol=1e-8, rtol=0, equal_nan=True):
    raise ValueError("The change table does not match the prevalence estimates.")
if not c.review.eq(first.review | last.review).all():
    raise ValueError("The change flags do not match the endpoint flags.")

if not BOUNDARY.is_file():
    try:
        with urlopen(URL, timeout=45) as response:
            raw = response.read()
        json.loads(raw)
    except (OSError, ValueError) as error:
        raise RuntimeError(f"Use the manual link above and save the boundary as {BOUNDARY.name}.") from error
    BOUNDARY.write_bytes(raw)
raw = BOUNDARY.read_bytes()
geo = json.loads(raw)
if geo.get("type") != "FeatureCollection":
    raise ValueError("Expected a GeoJSON FeatureCollection.")
features, boundary_rows = {}, []
for feature in geo["features"]:
    label = feature["properties"]["shapeName"]
    state = "FCT" if label == "Abuja Federal Capital Territory" else label
    if state in features:
        raise ValueError(f"Duplicate boundary name: {state}")
    features[state] = feature["geometry"]
    boundary_rows.append({"state": state, "boundary_label": label})
if set(features) != states:
    raise ValueError(f"Unmatched DHS names: {sorted(states - features.keys())}. "
                     f"Unmatched boundary names: {sorted(features.keys() - states)}.")

OUTPUT.mkdir(parents=True, exist_ok=True)
pd.DataFrame(boundary_rows).sort_values("state").to_csv(OUTPUT / "State_WASH_Boundary_Match.csv", index=False)
(OUTPUT / "Boundary_Source.txt").write_text(
    f"{CREDIT}\n{URL}\nSHA-256 {hashlib.sha256(raw).hexdigest()}\n", encoding="utf-8"
)
print(f"Matched {len(features)} areas. Checked {len(p)} prevalence and {len(c)} change rows.")


## 2. Draw the prevalence and change maps

Prevalence panels share a **0–100% scale**. The change maps use a common symmetric scale centred at zero. Hatching marks estimates flagged for review. Grey marks missing estimates. Consult the source CSVs for confidence intervals and the reason each estimate was flagged, especially for hygiene.


In [ ]:
dpi = 600

def save_figure(figure, filename):
    path = OUTPUT / filename
    figure.savefig(path, format="jpeg", dpi=dpi, facecolor="white",
                   pil_kwargs={"quality": 95, "subsampling": 0})
    plt.close(figure)
    with Image.open(path) as image:
        if image.format != "JPEG" or image.info.get("dpi") != (dpi, dpi):
            raise ValueError(f"Could not verify JPEG resolution for {path.name}.")
    return path



def polygon_path(geometry):
    kind = geometry["type"]
    if kind == "Polygon":
        polygons = [geometry["coordinates"]]
    elif kind == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        raise ValueError(f"Unexpected geometry type: {kind}")
    paths = []
    for polygon in polygons:
        for i, ring in enumerate(polygon):
            points = np.asarray(ring, dtype=float)[:, :2]
            if len(points) < 4 or not np.isfinite(points).all():
                raise ValueError("Malformed boundary ring.")
            if not np.array_equal(points[0], points[-1]):
                points = np.vstack([points, points[0]])
            x, y = points.T
            signed_area = np.sum(x[:-1] * y[1:] - y[:-1] * x[1:])
            # Opposite winding keeps interior holes open.
            if (signed_area > 0) != (i == 0):
                points = points[::-1]
            codes = np.full(len(points), PlotPath.LINETO, dtype=np.uint8)
            codes[0], codes[-1] = PlotPath.MOVETO, PlotPath.CLOSEPOLY
            paths.append(PlotPath(points, codes))
    return PlotPath.make_compound_path(*paths)

paths = {state: polygon_path(geometry) for state, geometry in features.items()}
coordinates = np.vstack([path.vertices for path in paths.values()])
west, south = coordinates.min(axis=0)
east, north = coordinates.max(axis=0)
if not (0 < west < east < 20 and 0 < south < north < 20):
    raise ValueError("Unexpected boundary extent. Use Nigeria ADM1 longitude and latitude.")
aspect = 1 / np.cos(np.deg2rad((south + north) / 2))

def draw_map(ax, frame, column, cmap, norm):
    for state, path in paths.items():
        row = frame.loc[state]
        value = row[column]
        color = cmap(norm(value)) if pd.notna(value) else "#D9DDE2"
        ax.add_patch(PathPatch(path, facecolor=color, edgecolor="#424952", linewidth=.35,
                               hatch="////" if pd.notna(value) and row.review else None))
    ax.set(xlim=(west - .2, east + .2), ylim=(south - .18, north + .18), aspect=aspect)
    ax.set_axis_off()

def map_key(fig, cmap, norm, ticks, label):
    bar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap),
                       cax=fig.add_axes([.23, .225, .54, .032]),
                       orientation="horizontal", ticks=ticks)
    bar.ax.tick_params(labelsize=8)
    bar.set_label(label, fontsize=8)
    handles = [Patch(facecolor="white", edgecolor="#424952", hatch="////", label="Flagged for review"),
               Patch(facecolor="#D9DDE2", edgecolor="#424952", label="Estimate missing")]
    fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(.5, .045),
               ncol=2, frameon=False, fontsize=8)
    fig.text(.5, .023, CREDIT, ha="center", va="bottom", fontsize=6.5, color="#40454A")

valid = c[DELTA].dropna()
limit = max(10, int(np.ceil(valid.abs().max() / 10)) * 10) if len(valid) else 10
blue, scale = plt.get_cmap("Blues"), Normalize(0, 100)
diverging, delta_scale = plt.get_cmap("RdBu"), TwoSlopeNorm(0, -limit, limit)
figures = {}
for indicator, label in NAMES.items():
    fig, axes = plt.subplots(1, 3, figsize=(7, 3.45))
    fig.subplots_adjust(left=.025, right=.975, top=.81, bottom=.29, wspace=.035)
    fig.suptitle(f"{label} by state", y=.97, fontsize=13, fontweight="bold")
    for ax, year in zip(axes, YEARS):
        draw_map(ax, p.xs((indicator, year), level=("indicator", "survey_round")),
                 "weighted_pct", blue, scale)
        ax.set_title(str(year), fontsize=10, pad=2)
    map_key(fig, blue, scale, [0, 25, 50, 75, 100], "Weighted household percentage")
    levels_path = save_figure(fig, f"Map_{indicator}_Prevalence_600dpi.jpg")

    fig, ax = plt.subplots(figsize=(7, 4.1))
    fig.subplots_adjust(left=.05, right=.95, top=.89, bottom=.27)
    fig.suptitle(f"{label} change by state", y=.97, fontsize=13, fontweight="bold")
    draw_map(ax, c.xs(indicator, level="indicator"), DELTA, diverging, delta_scale)
    map_key(fig, diverging, delta_scale, np.linspace(-limit, limit, 5), "2024 minus 2013 (percentage points)")
    change_path = save_figure(fig, f"Map_{indicator}_Change_600dpi.jpg")
    figures[indicator] = (levels_path, change_path)
print("Saved six JPEG maps with verified 600 × 600 dpi metadata.")


## 3. Save the report and map checks

The report includes all six maps and an editable summary of the mapped and flagged estimates. The notebook saves the DOCX first and then attempts Word 97–2003 DOC conversion. Outputs are in **wash_map_outputs**.


In [ ]:
from docx import Document
from docx.enum.table import WD_TABLE_ALIGNMENT, WD_CELL_VERTICAL_ALIGNMENT
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.shared import Inches, Pt, RGBColor

figure_width = 7
doc = Document()
doc.core_properties.title = "Maps of household WASH access in Nigeria"
doc.core_properties.author = ""
section = doc.sections[0]
section.page_width, section.page_height = Inches(8.5), Inches(11)
section.left_margin = section.right_margin = Inches(0.75)
section.top_margin, section.bottom_margin = Inches(0.7), Inches(0.7)

for style_name, size in [("Normal", 10.5), ("Title", 25), ("Heading 1", 17), ("Heading 2", 13), ("Caption", 9)]:
    style = doc.styles[style_name]
    style.font.name = "Calibri"
    style.font.size = Pt(size)
    style.font.color.rgb = RGBColor.from_string("000000")
    style.paragraph_format.space_before = Pt(5)
    style.paragraph_format.space_after = Pt(5)
    if style_name == "Title":
        border = style.element.get_or_add_pPr().find(qn("w:pBdr"))
        if border is not None:
            border.getparent().remove(border)

def xml(parent, tag, **attributes):
    element = OxmlElement(f"w:{tag}")
    for key, value in attributes.items():
        element.set(qn(f"w:{key}"), str(value))
    parent.append(element)
    return element

def add_table(caption, headers, rows, widths, text_columns, font_size=9.5):
    doc.add_paragraph(caption, style="Caption").paragraph_format.keep_with_next = True
    table = doc.add_table(rows=0, cols=len(headers))
    table.alignment = WD_TABLE_ALIGNMENT.CENTER
    table.autofit = False
    for column, width in zip(table.columns, widths):
        column.width = Inches(width)
    props = table._tbl.tblPr
    borders = xml(props, "tblBorders")
    for edge in ("top", "left", "bottom", "right", "insideH", "insideV"):
        xml(borders, edge, val="single", sz=4, color="D9D9D9")
    margins = xml(props, "tblCellMar")
    for edge, value in (("top", 16), ("bottom", 16), ("left", 85), ("right", 85)):
        xml(margins, edge, w=value, type="dxa")

    for i, values in enumerate([headers, *rows]):
        row = table.add_row()
        row_props = row._tr.get_or_add_trPr()
        xml(row_props, "cantSplit")
        if i == 0:
            xml(row_props, "tblHeader")
        for j, (cell, value, width) in enumerate(zip(row.cells, values, widths)):
            cell.width = Inches(width)
            cell.vertical_alignment = WD_CELL_VERTICAL_ALIGNMENT.CENTER
            if i == 0 or i % 2 == 0:
                xml(cell._tc.get_or_add_tcPr(), "shd", fill="203A50" if i == 0 else "F2F6F9")
            paragraph = cell.paragraphs[0]
            paragraph.alignment = WD_ALIGN_PARAGRAPH.LEFT if j in text_columns else WD_ALIGN_PARAGRAPH.CENTER
            paragraph.paragraph_format.space_after = Pt(0)
            paragraph.paragraph_format.space_before = Pt(0)
            paragraph.paragraph_format.line_spacing = 1.05
            run = paragraph.add_run(str(value))
            run.bold = i == 0
            run.font.size = Pt(font_size)
            if i == 0:
                run.font.color.rgb = RGBColor.from_string("FFFFFF")
    return table

def add_figure(path, caption, description):
    paragraph = doc.add_paragraph()
    paragraph.paragraph_format.space_before = Pt(7)
    paragraph.paragraph_format.keep_with_next = True
    paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER
    shape = paragraph.add_run().add_picture(str(path), width=Inches(figure_width))
    shape._inline.docPr.set("descr", description)
    doc.add_paragraph(caption, style="Caption")


qc_rows = []
for indicator, label in NAMES.items():
    for year in YEARS:
        frame = p.xs((indicator, year), level=("indicator", "survey_round"))
        known = frame.weighted_pct.notna()
        qc_rows.append([label, str(year), int(known.sum()), int((~known).sum()), int((known & frame.review).sum())])
    frame = c.xs(indicator, level="indicator")
    known = frame[DELTA].notna()
    qc_rows.append([label, "2024 minus 2013", int(known.sum()), int((~known).sum()), int((known & frame.review).sum())])
headers = ["Indicator", "Round or change", "Mapped", "Missing", "Flagged"]
pd.DataFrame(qc_rows, columns=headers).to_csv(OUTPUT / "State_WASH_Map_QC.csv", index=False)

doc.add_paragraph("Maps of household WASH access in Nigeria", style="Title")
doc.add_paragraph("Nigeria Demographic and Health Survey rounds 2013, 2018 and 2024")
doc.add_paragraph("The maps show weighted household prevalence and change across 36 states and the Federal Capital Territory. "
                  "All prevalence panels share a 0–100% scale. Change maps share a scale centred at zero.")
doc.add_heading("Map coverage and review flags", level=1)
add_table("Table 1. State and FCT estimates represented on the maps", headers, qc_rows,
          [1.80, 1.60, 1.05, 1.05, 1.50], {0, 1})
doc.add_paragraph("Hatching marks estimates flagged for review. Grey marks missing estimates. "
                  "The source CSVs provide confidence intervals, outcome coverage and the reason for each flag.")
doc.add_paragraph("Boundary source: GRID3 Nigeria ADM1 via geoBoundaries gbOpen. "
                  "The 2022 layer is used for all rounds. Boundary identifier NGA-ADM1-27671186. CC BY 4.0.")
for indicator, label in NAMES.items():
    doc.add_heading(label, level=1).paragraph_format.page_break_before = True
    levels_path, change_path = figures[indicator]
    add_figure(levels_path, "Weighted household prevalence in 2013, 2018 and 2024.",
               f"State maps of {label.lower()} prevalence with a common percentage scale.")
    add_figure(change_path, "Change in weighted household prevalence from 2013 to 2024.",
               f"State map of change in {label.lower()} between 2013 and 2024.")

def find_soffice():
    custom = os.environ.get("LIBREOFFICE_PATH")
    if custom and Path(custom).is_file():
        return custom
    candidates = [shutil.which("soffice"), shutil.which("libreoffice"),
                  "/Applications/LibreOffice.app/Contents/MacOS/soffice"]
    for variable in ("ProgramFiles", "ProgramFiles(x86)"):
        if base := os.environ.get(variable):
            candidates.append(str(Path(base) / "LibreOffice" / "program" / "soffice.com"))
    return next((candidate for candidate in candidates if candidate and Path(candidate).is_file()), None)


docx_path = OUTPUT / "Nigeria_DHS_WASH_Maps_Report.docx"
doc.save(docx_path)

def export_legacy_word(source, soffice):
    with TemporaryDirectory() as scratch:
        temp = Path(scratch).resolve()
        result = subprocess.run(
            [soffice, f"-env:UserInstallation={(temp / 'profile').as_uri()}", "--headless",
             "--convert-to", "doc:MS Word 97", "--outdir", str(temp), str(source.resolve())],
            capture_output=True, text=True, timeout=180,
        )
        converted = temp / source.with_suffix(".doc").name
        if result.returncode or not converted.is_file():
            raise RuntimeError(result.stderr or result.stdout or "Converter returned no file.")
        with converted.open("rb") as stream:
            if stream.read(8) != bytes.fromhex("D0CF11E0A1B11AE1"):
                raise ValueError("The converter did not create a binary Word 97–2003 document.")
        shutil.copyfile(converted, source.with_suffix(".doc"))

soffice = find_soffice()
if soffice:
    try:
        export_legacy_word(docx_path, soffice)
        print(f"Saved {docx_path.with_suffix('.doc').resolve()}")
    except (RuntimeError, ValueError, OSError, subprocess.TimeoutExpired) as error:
        print(f"Word .doc conversion failed: {error}")
        print("Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")
else:
    print("LibreOffice was unavailable. Open the saved .docx in desktop Word and save it as Word 97–2003 Document (*.doc).")

print(f"Saved the maps, report and map QC files to {OUTPUT.resolve()}.")
